In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.60.0 which is incompatible.


In [2]:
from google import genai

In [18]:
import os

os.environ["GEMINI_API_KEY"] =userdata.get("Gemini_Api_key")

In [13]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

In [5]:
categories = [
    "Payment Issue",
    "Refund Request",
    "Delivery Delay",
    "Account/Login Issue",
    "Product Issue",
    "Cancellation Request"
]

print(categories)

['Payment Issue', 'Refund Request', 'Delivery Delay', 'Account/Login Issue', 'Product Issue', 'Cancellation Request']


In [6]:
example_message = (
    "My payment was deducted, but my order was not confirmed."
)

example_category = "Payment Issue"

In [7]:
customer_message = "My card was charged twice for the same order."

print(customer_message)

My card was charged twice for the same order.


In [8]:
prompt = f"""
You are a customer-support classification assistant.

Classify the customer message into exactly ONE
of the following categories:

{", ".join(categories)}

ONE EXAMPLE:

Customer Message:
"{example_message}"

Category:
"{example_category}"

NOW CLASSIFY:

Customer Message:
"{customer_message}"

RULES:
- Return exactly one category.
- Use only the six allowed categories.
- Do not create a new category.
- Do not explain your answer.
- Return only the category name.
"""

In [9]:
print(prompt)


You are a customer-support classification assistant.

Classify the customer message into exactly ONE
of the following categories:

Payment Issue, Refund Request, Delivery Delay, Account/Login Issue, Product Issue, Cancellation Request

ONE EXAMPLE:

Customer Message:
"My payment was deducted, but my order was not confirmed."

Category:
"Payment Issue"

NOW CLASSIFY:

Customer Message:
"My card was charged twice for the same order."

RULES:
- Return exactly one category.
- Use only the six allowed categories.
- Do not create a new category.
- Do not explain your answer.
- Return only the category name.



In [19]:
# Re-initialize the client with the valid api_key variable from the kernel
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

In [20]:
result = response.text.strip()

print("Model Output:")
print(result)

Model Output:
Payment Issue


In [21]:
if result in categories:
    print("Valid Category:", result)
else:
    print("Invalid Category:", result)

Valid Category: Payment Issue


In [22]:
def classify_customer_message(customer_message):

    prompt = f"""
You are a customer-support classification assistant.

Classify the customer message into exactly ONE
of the following categories:

{", ".join(categories)}

ONE EXAMPLE:

Customer Message:
"{example_message}"

Category:
"{example_category}"

NOW CLASSIFY:

Customer Message:
"{customer_message}"

RULES:
- Return exactly one category.
- Use only the six allowed categories.
- Do not create a new category.
- Do not explain your answer.
- Return only the category name.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    result = response.text.strip()

    if result in categories:
        return result
    else:
        return "Invalid Category"

In [23]:
result = classify_customer_message(
    "My card was charged twice for the same order."
)

print(result)

Payment Issue


In [25]:
test_messages = [
    "I was charged twice for the same order.",
    "I want my money back for this product.",
    "My package has not arrived yet.",
    "I cannot login to my account.",
    "The product I received is damaged."
]

In [26]:
for message in test_messages:

    result = classify_customer_message(message)

    print("Customer Message:")
    print(message)

    print("Predicted Category:")
    print(result)

    print("-" * 50)

Customer Message:
I was charged twice for the same order.
Predicted Category:
Payment Issue
--------------------------------------------------
Customer Message:
I want my money back for this product.
Predicted Category:
Refund Request
--------------------------------------------------
Customer Message:
My package has not arrived yet.
Predicted Category:
Delivery Delay
--------------------------------------------------
Customer Message:
I cannot login to my account.
Predicted Category:
Account/Login Issue
--------------------------------------------------
Customer Message:
The product I received is damaged.
Predicted Category:
Product Issue
--------------------------------------------------


In [27]:
expected_categories = [
    "Payment Issue",
    "Refund Request",
    "Delivery Delay",
    "Account/Login Issue",
    "Product Issue"
]

In [28]:
correct = 0

for message, expected in zip(
    test_messages,
    expected_categories
):

    predicted = classify_customer_message(message)

    print("Message:", message)
    print("Expected:", expected)
    print("Predicted:", predicted)

    if predicted == expected:
        correct += 1

    print("-" * 50)

accuracy = (correct / len(test_messages)) * 100

print("Accuracy:", accuracy, "%")

Message: I was charged twice for the same order.
Expected: Payment Issue
Predicted: Payment Issue
--------------------------------------------------
Message: I want my money back for this product.
Expected: Refund Request
Predicted: Refund Request
--------------------------------------------------
Message: My package has not arrived yet.
Expected: Delivery Delay
Predicted: Delivery Delay
--------------------------------------------------
Message: I cannot login to my account.
Expected: Account/Login Issue
Predicted: Account/Login Issue
--------------------------------------------------
Message: The product I received is damaged.
Expected: Product Issue
Predicted: Product Issue
--------------------------------------------------
Accuracy: 100.0 %


In [29]:
improved_prompt = f"""
You are a customer-support classification assistant.

Classify the customer message into exactly ONE
of these six categories:

1. Payment Issue
2. Refund Request
3. Delivery Delay
4. Account/Login Issue
5. Product Issue
6. Cancellation Request

CATEGORY DEFINITIONS:

Payment Issue:
Problems involving payments, card charges,
duplicate charges, or payment confirmation.

Refund Request:
The customer wants their money back.

Delivery Delay:
The customer's package or order is delayed
or has not arrived.

Account/Login Issue:
Problems with login or account access.

Product Issue:
Problems with the product itself,
such as damage or defects.

Cancellation Request:
The customer wants to cancel an order.

ONE EXAMPLE:

Customer Message:
"{example_message}"

Category:
"{example_category}"

NOW CLASSIFY:

Customer Message:
"{customer_message}"

STRICT OUTPUT RULES:

- Return exactly ONE category.
- Use only the six categories listed above.
- Do not invent a category.
- Do not explain.
- Return only the category name.
"""

In [30]:
import gradio as gr

In [31]:
demo = gr.Interface(
    fn=classify_customer_message,

    inputs=gr.Textbox(
        label="💬 Customer Message",
        placeholder="Example: My card was charged twice for the same order.",
        lines=5
    ),

    outputs=gr.Textbox(
        label="🤖 Predicted Category"
    ),

    title="🤖 Customer Support AI",

    description=(
        "One-Shot Prompting Based Customer Support Classification"
    ),

    examples=[
        ["My card was charged twice for the same order."],
        ["I want my money back for this product."],
        ["My package has not arrived yet."],
        ["I cannot login to my account."],
        ["The product I received is damaged."]
    ]
)

In [32]:
demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71ac662767335e8b32.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
